# Training Experiments — SigLIP-B/16-384 + LoRA

Scratch notebook for trying training ideas quickly.

**To run a new experiment:** change the config in cell 2, then Run All.  
**B0 frozen baseline** (1.43% mIoU) is fixed — see `report.md` and `03_siglip_b0_eval.ipynb`.  
Results are logged to W&B project `region-grounded`.

Trains **M_human** only: L_global (MHAP pooler) + λ·L_region with human Flickr30k Entities bboxes.

| Cell | Purpose |
|------|---------|
| 0–2  | Config — **edit here** |
| 3–7  | Infrastructure (data, model, losses, helpers) |
| 8    | `quick_voc_eval` |
| 9    | `train_one_epoch` + optimizer/scheduler |
| 10+  | Experiment run |

In [1]:
# ── 0. Install dependencies ──────────────────────────────────────────────────
# Run once at the start of a Colab session.
import subprocess, sys
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'peft>=0.10', 'datasets', 'transformers>=4.40', 'huggingface_hub',
    'tqdm', 'Pillow', 'wandb', 'torchao>=0.16.0',
], check=True)
print('Dependencies installed.')

Dependencies installed.


In [2]:
# ── 1. Imports ─────────────\───────────────────────────────────────────────────
import os, sys, io, random, getpass, zipfile, urllib.request, shutil
from pathlib import Path
from copy import deepcopy

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image as PILImage
from tqdm.auto import tqdm
import torchvision.datasets as tvd

from transformers import SiglipModel, AutoProcessor
from datasets import load_dataset
from huggingface_hub import HfApi, hf_hub_download
from peft import get_peft_model, LoraConfig
import wandb

if not os.environ.get('HF_TOKEN'):
    os.environ['HF_TOKEN'] = getpass.getpass('HF token: ')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU  : {torch.cuda.get_device_name(0)}')
    print(f'VRAM : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

Device: cuda
GPU  : NVIDIA L4
VRAM : 23.7 GB


In [3]:
# ── 2. Config ─────────────────────────────────────────────────────────────────
# ← EDIT THIS CELL to change the experiment.

CFG = dict(
    model_id       = 'google/siglip-base-patch16-384',
    eval_size      = 384,
    patch_size     = 16,

    # LoRA
    lora_rank      = 4,
    lora_alpha     = 16,
    lora_dropout   = 0.1,
    lora_targets   = ['q_proj', 'v_proj'],

    # Training
    batch_size     = 32,
    region_batch   = 64,       # phrase-bbox pairs per region loss step
    lr             = 2e-4,
    weight_decay   = 0.01,
    epochs         = 1,
    warmup_steps   = 100,
    prompt         = 'a photo of a {}',

    # Region loss weight  (>0 = global + region)
    lambda_region  = 0.5,

    # Eval
    eval_images    = 200,
    tau_seg        = 0.0,

    ckpt_dir       = 'checkpoints',
)
CFG['n_side'] = CFG['eval_size'] // CFG['patch_size']  # 24
Path(CFG['ckpt_dir']).mkdir(exist_ok=True)

# Known frozen baseline — do not recompute here; see report.md
B0_MIOU = 1.43

print(CFG)
print(f'B0 baseline (frozen, from report.md): {B0_MIOU}%')

{'model_id': 'google/siglip-base-patch16-384', 'eval_size': 384, 'patch_size': 16, 'lora_rank': 4, 'lora_alpha': 16, 'lora_dropout': 0.1, 'lora_targets': ['q_proj', 'v_proj'], 'batch_size': 32, 'region_batch': 64, 'lr': 0.0002, 'weight_decay': 0.01, 'epochs': 1, 'warmup_steps': 100, 'prompt': 'a photo of a {}', 'lambda_region': 0.5, 'eval_images': 200, 'tau_seg': 0.0, 'ckpt_dir': 'checkpoints', 'n_side': 24}
B0 baseline (frozen, from report.md): 1.43%


In [4]:
# ── 2b. W&B init ─────────────────────────────────────────────────────────────
wandb.login()

run = wandb.init(
    project = 'region-grounded',
    name    = f'mhuman_lam{CFG["lambda_region"]}_r{CFG["lora_rank"]}',
    config  = CFG,
    tags    = ['siglip', 'flickr30k', 'lora'],
)
print(f'W&B run: {run.url}')

wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: sidraj (sidraj-university-of-chicago-charter-school) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


W&B run: https://wandb.ai/sidraj-university-of-chicago-charter-school/region-grounded/runs/8b71jyl9


In [5]:
# ── 3. Data — load from Drive cache or download fresh ───────────────────────
# First run  : downloads parquet + entities, zips, uploads to Drive (~15 min).
# Later runs : mounts Drive, extracts the zip locally (~2 min).

from google.colab import drive

GDRIVE_ROOT  = Path('/content/drive')
drive.mount(str(GDRIVE_ROOT))

GDRIVE_ZIP   = GDRIVE_ROOT / 'MyDrive' / 'region-grounded-data' / 'flickr30k_full.zip'
LOCAL_DATA   = Path('data/flickr30k')
PQ_DIR       = LOCAL_DATA / 'parquet'
ENTITIES_DIR = LOCAL_DATA / 'entities'
ANN_DIR      = ENTITIES_DIR / 'Annotations'
SENT_DIR     = ENTITIES_DIR / 'Sentences'

if GDRIVE_ZIP.exists():
    # ── Fast path ─────────────────────────────────────────────────────────────
    sz_gb = GDRIVE_ZIP.stat().st_size / 1e9
    print(f'Drive cache found ({sz_gb:.1f} GB). Extracting...')
    LOCAL_DATA.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(str(GDRIVE_ZIP)) as zf:
        zf.extractall('.')
    print('Extracted.')

else:
    # ── First run: download everything, then zip to Drive ─────────────────────
    PQ_DIR.mkdir(parents=True, exist_ok=True)
    ENTITIES_DIR.mkdir(parents=True, exist_ok=True)

    # 1. Flickr30k parquet shards
    REPO_ID  = 'nlphuji/flickr30k'
    api      = HfApi(token=os.environ['HF_TOKEN'])
    pq_names = sorted(
        f for f in api.list_repo_files(REPO_ID, repo_type='dataset',
                                       revision='refs/convert/parquet')
        if f.endswith('.parquet')
    )
    print(f'Downloading {len(pq_names)} parquet shard(s)...')
    for fname in pq_names:
        cached = hf_hub_download(
            repo_id=REPO_ID, filename=fname, repo_type='dataset',
            revision='refs/convert/parquet', token=os.environ['HF_TOKEN'],
        )
        shutil.copy(cached, PQ_DIR / Path(fname).name)
    print(f'Parquet shards → {PQ_DIR}')

    # 2. Flickr30k Entities utils + annotations
    UTILS_URL = ('https://raw.githubusercontent.com/bryanplummer/'
                 'flickr30k_entities/master/flickr30k_entities_utils.py')
    urllib.request.urlretrieve(UTILS_URL, ENTITIES_DIR / 'flickr30k_entities_utils.py')

    ZIP_URL   = ('https://github.com/bryanplummer/flickr30k_entities/'
                 'raw/master/annotations.zip')
    TMP_ANN   = Path('/tmp/annotations.zip')
    if not ANN_DIR.is_dir():
        print('Downloading Entities annotations (~25 MB)...')
        urllib.request.urlretrieve(ZIP_URL, TMP_ANN)
        with zipfile.ZipFile(TMP_ANN) as zf:
            zf.extractall(ENTITIES_DIR)
        print('Extracted.')

    # 3. Zip all data and save to Drive (ZIP_STORED: parquet is already compressed)
    TMP_ZIP = Path('/tmp/flickr30k_full.zip')
    total   = sum(1 for f in LOCAL_DATA.rglob('*') if f.is_file())
    print(f'Zipping {total} files (no compression — parquet is pre-compressed)...')
    with zipfile.ZipFile(str(TMP_ZIP), 'w', zipfile.ZIP_STORED) as zf:
        for f in LOCAL_DATA.rglob('*'):
            if f.is_file():
                zf.write(str(f), str(f))
    sz_gb = TMP_ZIP.stat().st_size / 1e9
    GDRIVE_ZIP.parent.mkdir(parents=True, exist_ok=True)
    print(f'Uploading {sz_gb:.1f} GB to Drive...')
    shutil.copy(str(TMP_ZIP), str(GDRIVE_ZIP))
    TMP_ZIP.unlink()
    print(f'Saved to {GDRIVE_ZIP}')

# Load parquet into HuggingFace Dataset
local_pq = sorted(str(f) for f in PQ_DIR.glob('*.parquet'))
hf_data  = load_dataset('parquet', data_files={'data': local_pq})['data']
print(f'Flickr30k rows: {len(hf_data):,}')
print('Columns:', hf_data.column_names)

# Import Entities utils
if str(ENTITIES_DIR) not in sys.path:
    sys.path.insert(0, str(ENTITIES_DIR))
from flickr30k_entities_utils import get_sentence_data, get_annotations

print(f'Annotations: {len(list(ANN_DIR.iterdir())):,} XML files')
print(f'Sentences  : {len(list(SENT_DIR.iterdir())):,} txt files')


Mounted at /content/drive
Drive cache found (4.4 GB). Extracting...
Extracted.


Generating data split: 0 examples [00:00, ? examples/s]

Flickr30k rows: 31,014
Columns: ['image', 'caption', 'sentids', 'split', 'img_id', 'filename']
Annotations: 31,783 XML files
Sentences  : 31,783 txt files


In [6]:
# ── 4. Dataset ────────────────────────────────────────────────────────────────
# One item per image. Returns:
#   pil_image  : PIL image (original size, unprocessed)
#   caption    : one randomly sampled caption string
#   orig_size  : (W, H) of the original image — needed for bbox scaling
#   phrase_boxes: list of (phrase_str, [x1,y1,x2,y2]) in original pixel coords

class FlickrSigLIPDataset(Dataset):

    def __init__(self, hf_data, ann_dir: Path, sent_dir: Path, split: str = 'train'):
        self.ann_dir  = ann_dir
        self.sent_dir = sent_dir
        self.rows     = [r for r in hf_data if r['split'] == split]

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        row = self.rows[idx]

        # decode PIL image
        img_field = row['image']
        if isinstance(img_field, PILImage.Image):
            pil = img_field.convert('RGB')
        else:
            pil = PILImage.open(io.BytesIO(img_field['bytes'])).convert('RGB')
        orig_size = pil.size  # (W, H)

        caption = random.choice(row['caption'])

        # phrase-bbox pairs from Entities annotations
        stem = row['filename'].replace('.jpg', '')
        try:
            anns  = get_annotations(str(self.ann_dir  / f'{stem}.xml'))
            sents = get_sentence_data(str(self.sent_dir / f'{stem}.txt'))
        except Exception:
            return pil, caption, orig_size, []

        phrase_boxes = []
        seen = set()
        for sent in sents:
            for phrase in sent['phrases']:
                pid = phrase['phrase_id']
                if pid in seen or pid not in anns['boxes']:
                    continue
                seen.add(pid)
                for box in anns['boxes'][pid]:
                    phrase_boxes.append((phrase['phrase'], box))

        return pil, caption, orig_size, phrase_boxes

    @staticmethod
    def collate_fn(batch):
        pils, captions, orig_sizes, phrase_boxes = zip(*batch)
        return list(pils), list(captions), list(orig_sizes), list(phrase_boxes)


train_ds = FlickrSigLIPDataset(hf_data, ANN_DIR, SENT_DIR, split='train')
val_ds   = FlickrSigLIPDataset(hf_data, ANN_DIR, SENT_DIR, split='val')
print(f'Train: {len(train_ds):,} images')
print(f'Val  : {len(val_ds):,} images')

# Sanity check one sample
pil0, cap0, sz0, pb0 = train_ds[0]
print(f'\nSample: size={sz0}  caption="{cap0[:60]}…"')
print(f'phrase-box pairs: {len(pb0)}  e.g. {pb0[0] if pb0 else "none"}')

Train: 29,000 images
Val  : 1,014 images

Sample: size=(333, 500)  caption="Two men in green shirts are standing in a yard.…"
phrase-box pairs: 12  e.g. ('Two young guys', [158, 124, 218, 334])


In [7]:
# ── 5. Load SigLIP + apply LoRA ───────────────────────────────────────────────
processor = AutoProcessor.from_pretrained(CFG['model_id'],
                                          token=os.environ['HF_TOKEN'])
base_model = SiglipModel.from_pretrained(CFG['model_id'],
                                         token=os.environ['HF_TOKEN'])

lora_cfg = LoraConfig(
    r                = CFG['lora_rank'],
    lora_alpha       = CFG['lora_alpha'],
    lora_dropout     = CFG['lora_dropout'],
    target_modules   = CFG['lora_targets'],
    bias             = 'none',
)
model = get_peft_model(base_model, lora_cfg)
model = model.to(DEVICE)
model.print_trainable_parameters()

# logit_scale and logit_bias are SigLIP's contrastive temperature parameters.
# Keep them trainable — they are NOT in LoRA modules.
for n, p in model.named_parameters():
    if 'logit_scale' in n or 'logit_bias' in n:
        p.requires_grad_(True)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Trainable: {trainable:,} / {total:,}  ({100*trainable/total:.2f}%)')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/368 [00:00<?, ?B/s]

The image processor of type `SiglipImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json:   0%|          | 0.00/322 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/711 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/409 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/814M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/408 [00:00<?, ?it/s]

trainable params: 294,912 || all params: 203,742,722 || trainable%: 0.1447
Trainable: 294,914 / 203,742,722  (0.14%)


In [8]:
# ── 6. Loss functions ─────────────────────────────────────────────────────────

def siglip_global_loss(img_feats, txt_feats, logit_scale, logit_bias):
    logits = logit_scale.exp() * (img_feats @ txt_feats.T) + logit_bias
    B      = logits.shape[0]
    labels = 2 * torch.eye(B, device=logits.device) - 1
    return -F.logsigmoid(labels * logits).mean()


def bbox_patch_mask(bbox, orig_size, n_side, eval_size):
    W, H = orig_size
    x1, y1, x2, y2 = bbox
    sx, sy = eval_size / W, eval_size / H
    x1, y1, x2, y2 = x1*sx, y1*sy, x2*sx, y2*sy
    patch_size = eval_size / n_side
    rows = torch.arange(n_side, dtype=torch.float32)
    cols = torch.arange(n_side, dtype=torch.float32)
    cy = (rows + 0.5) * patch_size
    cx = (cols + 0.5) * patch_size
    grid_cy, grid_cx = torch.meshgrid(cy, cx, indexing='ij')
    mask = (grid_cx >= x1) & (grid_cx <= x2) & (grid_cy >= y1) & (grid_cy <= y2)
    return mask.reshape(-1)


def filip_region_loss(patch_feats_n, phrase_token_feats, bbox_masks,
                      logit_scale, logit_bias):
    """
    Vectorised FILIP-style region-phrase contrastive loss.

    patch_feats_n    : (B, N, D) L2-normalised patch features
    phrase_token_feats: list of B tensors, each (T_i, D) L2-normalised token feats
    bbox_masks       : (B, N) bool — which patches belong to each item's bbox

    Replaces the O(B²) Python loop with a single (B·T_max, D) × (D, B·K_max)
    matmul, keeping peak memory ~49 MB at B=128, T=15, K=100, D=768.
    """
    B, N, D = patch_feats_n.shape
    device  = patch_feats_n.device
    dtype   = patch_feats_n.dtype

    # ── Pad phrase tokens → (B, T_max, D) ────────────────────────────────────
    T_max = max(t.shape[0] for t in phrase_token_feats)
    tokens_pad  = patch_feats_n.new_zeros(B, T_max, D)
    phrase_mask = torch.zeros(B, T_max, dtype=torch.bool, device=device)
    for i, t in enumerate(phrase_token_feats):
        tokens_pad[i, :t.shape[0]]  = t
        phrase_mask[i, :t.shape[0]] = True

    # ── Extract + pad bbox patches → (B, K_max, D) ───────────────────────────
    bbox_patches = [patch_feats_n[j][bbox_masks[j]] for j in range(B)]
    K_max = max(max(p.shape[0] for p in bbox_patches), 1)
    patches_pad = patch_feats_n.new_zeros(B, K_max, D)
    patch_mask  = torch.zeros(B, K_max, dtype=torch.bool, device=device)
    for j, p in enumerate(bbox_patches):
        if p.shape[0] > 0:
            patches_pad[j, :p.shape[0]] = p
            patch_mask[j, :p.shape[0]]  = True

    # ── One matmul → (B, T_max, B, K_max) ────────────────────────────────────
    sim = (tokens_pad.reshape(B * T_max, D) @
           patches_pad.reshape(B * K_max, D).T
           ).reshape(B, T_max, B, K_max)

    # mask padding patches with -inf before max
    sim = sim.masked_fill(~patch_mask[None, None], float('-inf'))

    # max over patches → (B, T_max, B); zero out empty bboxes
    max_sim   = sim.amax(dim=-1)
    empty_box = ~patch_mask.any(dim=-1)                          # (B,)
    max_sim   = max_sim.masked_fill(empty_box[None, None, :], 0.0)

    # zero out padding tokens; mean over valid tokens → (B, B) scores
    max_sim = max_sim.masked_fill(~phrase_mask[:, :, None], 0.0)
    n_toks  = phrase_mask.sum(dim=1).clamp(min=1).to(dtype)      # (B,)
    scores  = max_sim.sum(dim=1) / n_toks[:, None]               # (B, B)

    logits = logit_scale.exp() * scores + logit_bias
    labels = 2 * torch.eye(B, device=device) - 1
    return -F.logsigmoid(labels * logits).mean()


print('Loss functions defined: siglip_global_loss, filip_region_loss')

Loss functions defined: siglip_global_loss, filip_region_loss


In [ ]:
# ── 7. Feature extraction helpers ─────────────────────────────────────────────
import types


def _maskclip_attn_fwd(self, hidden_states, attention_mask=None, **kwargs):
    """MaskCLIP-style bypass for the last attention layer: replace full
    self-attention with out_proj(v_proj(hidden_states)). When installed
    permanently during training, the gradient flows through the same patch
    representation that PG / R@1 / VOC eval reads — fixes the v1 mismatch
    where training used full attention but metrics used v_proj-only.
    """
    return self.out_proj(self.v_proj(hidden_states)), None


def install_maskclip_bypass(model):
    """Swap the last vision attention layer's forward to the MaskCLIP bypass.
    Returns the original forward so it can be restored after training.
    Safe under gradient checkpointing: the bypass stays installed for the
    rematerialised forward during backward, so the autograd graph stays
    consistent."""
    last_attn = model.vision_model.encoder.layers[-1].self_attn
    orig_fwd  = last_attn.forward
    last_attn.forward = types.MethodType(_maskclip_attn_fwd, last_attn)
    return orig_fwd


def restore_attn(model, orig_fwd):
    """Undo install_maskclip_bypass."""
    model.vision_model.encoder.layers[-1].self_attn.forward = orig_fwd


def get_image_feats(model, pixel_values):
    """Global image embedding via MHAP pooler_output, L2-normalised.
    Uses pooler_output so the global loss gradient does not collapse patch tokens.
    """
    out = model.vision_model(pixel_values=pixel_values)
    return F.normalize(out.pooler_output, dim=-1)   # (B, D)


def get_patch_feats(model, pixel_values):
    """Per-patch features after post_layernorm, L2-normalised. Shape (B, N, D).
    Caller is expected to have installed the MaskCLIP bypass on the last
    attention layer (see install_maskclip_bypass) so train and eval read
    the same patch representation."""
    out = model.vision_model(pixel_values=pixel_values)
    return F.normalize(out.last_hidden_state, dim=-1)   # (B, N, D)


def get_text_feats(model, input_ids, attention_mask=None):
    """Global text embedding: EOS token, L2-normalised. Shape (B, D)."""
    out = model.text_model(input_ids=input_ids, attention_mask=attention_mask)
    return F.normalize(out.last_hidden_state[:, -1, :], dim=-1)   # (B, D)


def get_phrase_token_feats(model, processor, phrases, device):
    """Per-token text features for a list of phrases. Returns list of (T_i, D) tensors."""
    inputs = processor(text=phrases, return_tensors='pt',
                       padding=True, truncation=True).to(device)
    out    = model.text_model(**inputs)
    hs     = out.last_hidden_state   # (B, T, D)
    result = []
    for i, phrase in enumerate(phrases):
        if 'attention_mask' in inputs:
            length = inputs['attention_mask'][i].sum().item()
        else:
            length = hs.shape[1]
        tok_feats = F.normalize(hs[i, :length, :], dim=-1)  # (T, D)
        result.append(tok_feats)
    return result


print('Feature extraction helpers defined (with MaskCLIP bypass).')


In [10]:
# ── 8. Quick VOC eval ─────────────────────────────────────────────────────────
# Reuses the same eval pipeline as 02_b0_benchmark (MaskCLIP-style extraction).

VOC_CLASSES = [
    'aeroplane', 'bicycle', 'bird', 'boat', 'bottle',
    'bus', 'car', 'cat', 'chair', 'cow',
    'diningtable', 'dog', 'horse', 'motorbike', 'person',
    'pottedplant', 'sheep', 'sofa', 'train', 'tvmonitor',
]

VOC_ROOT = Path('/tmp/voc')
VOC_ROOT.mkdir(exist_ok=True)
voc_val  = tvd.VOCSegmentation(root=str(VOC_ROOT), year='2012',
                                image_set='val', download=True)
print(f'VOC 2012 val: {len(voc_val)} images')


def _maskclip_attn_fwd(self, hidden_states, attention_mask=None, **kwargs):
    return self.out_proj(self.v_proj(hidden_states)), None


def quick_voc_eval(model, processor, n_images=200, tau_seg=0.0, device=DEVICE):
    """Run MaskCLIP-style zero-shot segmentation on n_images VOC val images."""
    model.eval()
    n_fg   = len(VOC_CLASSES)
    n_cls  = n_fg + 1

    # Encode text classes once
    prompts = [f'a photo of a {c}' for c in VOC_CLASSES]
    txt_in  = processor(text=prompts, return_tensors='pt',
                        padding=True).to(device)
    with torch.no_grad():
        txt_feats = F.normalize(
            model.text_model(**txt_in).last_hidden_state[:, -1, :], dim=-1
        )   # (20, D)

    # Patch extraction with MaskCLIP-style last attention bypass.
    # Patching self_attn.forward means model.vision_model(...) will use the bypass
    # and still apply post_layernorm, so last_hidden_state is correctly normalised.
    last_attn = model.vision_model.encoder.layers[-1].self_attn
    orig_fwd  = last_attn.forward
    last_attn.forward = types.MethodType(_maskclip_attn_fwd, last_attn)

    tp = torch.zeros(n_cls, dtype=torch.float64)
    fp = torch.zeros(n_cls, dtype=torch.float64)
    fn = torch.zeros(n_cls, dtype=torch.float64)
    gt_present = torch.zeros(n_cls, dtype=torch.bool)

    try:
        for i in tqdm(range(n_images), desc='VOC eval', leave=False):
            pil_img, target = voc_val[i]
            orig_w, orig_h  = pil_img.size
            gt = torch.from_numpy(np.array(target)).long()

            pix = processor(images=pil_img, return_tensors='pt').pixel_values.to(device)
            with torch.no_grad():
                out = model.vision_model(pixel_values=pix)

            # last_hidden_state is after post_layernorm — matches text feature space
            patch_n = F.normalize(out.last_hidden_state[0], dim=-1)   # (N, D)
            sim     = patch_n @ txt_feats.T                            # (N, 20)
            n_side  = int(sim.shape[0] ** 0.5)
            sim_up  = F.interpolate(
                sim.reshape(n_side, n_side, n_fg).permute(2,0,1).unsqueeze(0).float(),
                size=(orig_h, orig_w), mode='bilinear', align_corners=False
            ).squeeze(0).permute(1, 2, 0)

            max_sim, pred = sim_up.max(dim=-1)
            pred = pred + 1
            pred[max_sim < tau_seg] = 0
            pred = pred.cpu()

            valid = (gt != 255)
            pv, gv = pred[valid], gt[valid]
            for c in range(n_cls):
                pc = (pv == c); gc = (gv == c)
                tp[c] += (pc & gc).sum()
                fp[c] += (pc & ~gc).sum()
                fn[c] += (~pc & gc).sum()
                if gc.any(): gt_present[c] = True
    finally:
        last_attn.forward = orig_fwd

    iou   = tp / (tp + fp + fn).clamp(min=1e-6)
    miou  = iou[gt_present].mean().item() * 100
    return miou


print('quick_voc_eval defined.')

100%|██████████| 2.00G/2.00G [01:33<00:00, 21.4MB/s] 


VOC 2012 val: 1449 images
quick_voc_eval defined.


In [ ]:
# ── 8b. Quick Pointing Game + Recall@1 eval (Flickr30k) ──────────────────────
# Same protocol as 06_pointing_game_eval.ipynb so trained-model numbers are
# directly comparable to B0 = 14.74% (PG) / 6.83% (R@1 top-10) / 7.91% (top-25).

_pg_val_rows = None

def _get_pg_val_rows():
    global _pg_val_rows
    if _pg_val_rows is None:
        _pg_val_rows = [r for r in hf_data if r['split'] == 'val']
    return _pg_val_rows


def _load_phrase_boxes(row):
    stem = row['filename'].replace('.jpg', '')
    try:
        anns  = get_annotations(str(ANN_DIR  / f'{stem}.xml'))
        sents = get_sentence_data(str(SENT_DIR / f'{stem}.txt'))
    except Exception:
        return []
    pairs, seen = [], set()
    for sent in sents:
        for phrase in sent['phrases']:
            pid = phrase['phrase_id']
            if pid in seen or pid not in anns['boxes']:
                continue
            seen.add(pid)
            for box in anns['boxes'][pid]:
                pairs.append((phrase['phrase'], box))
    return pairs


def _row_to_pil(row):
    img = row['image']
    if isinstance(img, PILImage.Image):
        return img.convert('RGB')
    return PILImage.open(io.BytesIO(img['bytes'])).convert('RGB')


def quick_pointing_game_eval(model, processor, n_images=200, seed=42, device=DEVICE):
    """Flickr30k Pointing Game on val split. MaskCLIP-style last-attention bypass.
    Returns (acc_pct, n_correct, n_total)."""
    model.eval()
    rows = _get_pg_val_rows()
    rng  = random.Random(seed)
    rows = rng.sample(rows, min(n_images, len(rows)))

    n_side    = CFG['n_side']
    eval_size = CFG['eval_size']
    patch_px  = eval_size / n_side

    last_attn = model.vision_model.encoder.layers[-1].self_attn
    orig_fwd  = last_attn.forward
    last_attn.forward = types.MethodType(_maskclip_attn_fwd, last_attn)

    n_correct = n_total = 0
    try:
        for row in tqdm(rows, desc='PG eval', leave=False):
            phrase_boxes = _load_phrase_boxes(row)
            if not phrase_boxes:
                continue
            pil = _row_to_pil(row)
            W, H = pil.size

            pix = processor(images=pil, return_tensors='pt').pixel_values.to(device)
            with torch.no_grad():
                out = model.vision_model(pixel_values=pix)
            patch_feats = F.normalize(out.last_hidden_state[0], dim=-1)

            for phrase, bbox in phrase_boxes:
                x1, y1, x2, y2 = bbox
                if x2 <= x1 or y2 <= y1:
                    continue
                txt_in = processor(text=[phrase], return_tensors='pt',
                                   padding=True, truncation=True).to(device)
                with torch.no_grad():
                    txt_hs = model.text_model(**txt_in).last_hidden_state
                txt_feat = F.normalize(txt_hs[0, -1, :], dim=-1)

                sim  = patch_feats @ txt_feat
                best = sim.argmax().item()
                pr, pc = divmod(best, n_side)
                cx = (pc + 0.5) * patch_px * W / eval_size
                cy = (pr + 0.5) * patch_px * H / eval_size
                if x1 <= cx <= x2 and y1 <= cy <= y2:
                    n_correct += 1
                n_total += 1
    finally:
        last_attn.forward = orig_fwd

    return 100.0 * n_correct / max(n_total, 1), n_correct, n_total


def _patches_to_box(indices, n_side, patch_px, W, H, eval_size):
    rs = np.array([i // n_side for i in indices])
    cs = np.array([i %  n_side for i in indices])
    x1 = float(cs.min()       * patch_px * W / eval_size)
    y1 = float(rs.min()       * patch_px * H / eval_size)
    x2 = float((cs.max() + 1) * patch_px * W / eval_size)
    y2 = float((rs.max() + 1) * patch_px * H / eval_size)
    return x1, y1, x2, y2


def _iou(a, b):
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b
    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    inter = max(0, ix2 - ix1) * max(0, iy2 - iy1)
    if inter == 0:
        return 0.0
    union = (ax2-ax1)*(ay2-ay1) + (bx2-bx1)*(by2-by1) - inter
    return inter / union if union > 0 else 0.0


def quick_recall1_eval(model, processor, n_images=200, seed=42,
                        iou_thresh=0.5, device=DEVICE):
    """Flickr30k Recall@1 @ IoU >= iou_thresh on val. Returns dict for top10/top25/halfmax."""
    model.eval()
    rows = _get_pg_val_rows()
    rng  = random.Random(seed)
    rows = rng.sample(rows, min(n_images, len(rows)))

    n_side    = CFG['n_side']
    eval_size = CFG['eval_size']
    patch_px  = eval_size / n_side

    last_attn = model.vision_model.encoder.layers[-1].self_attn
    orig_fwd  = last_attn.forward
    last_attn.forward = types.MethodType(_maskclip_attn_fwd, last_attn)

    hits   = {'top10': 0, 'top25': 0, 'halfmax': 0}
    totals = {'top10': 0, 'top25': 0, 'halfmax': 0}

    try:
        for row in tqdm(rows, desc='R@1 eval', leave=False):
            phrase_boxes = _load_phrase_boxes(row)
            if not phrase_boxes:
                continue
            pil = _row_to_pil(row)
            W, H = pil.size

            pix = processor(images=pil, return_tensors='pt').pixel_values.to(device)
            with torch.no_grad():
                out = model.vision_model(pixel_values=pix)
            patch_feats = F.normalize(out.last_hidden_state[0], dim=-1)

            for phrase, gt in phrase_boxes:
                x1, y1, x2, y2 = gt
                if x2 <= x1 or y2 <= y1:
                    continue
                txt_in = processor(text=[phrase], return_tensors='pt',
                                   padding=True, truncation=True).to(device)
                with torch.no_grad():
                    txt_hs = model.text_model(**txt_in).last_hidden_state
                txt_feat = F.normalize(txt_hs[0, -1, :], dim=-1)
                sim = (patch_feats @ txt_feat).cpu().numpy()

                strategies = {
                    'top10':   np.argsort(sim)[-10:][::-1],
                    'top25':   np.argsort(sim)[-25:][::-1],
                    'halfmax': np.where(sim >= 0.5 * sim.max())[0],
                }
                for name, idxs in strategies.items():
                    totals[name] += 1
                    if len(idxs) == 0:
                        continue
                    pred = _patches_to_box(list(idxs), n_side, patch_px, W, H, eval_size)
                    if _iou(pred, gt) >= iou_thresh:
                        hits[name] += 1
    finally:
        last_attn.forward = orig_fwd

    return {k: (100.0 * hits[k] / max(totals[k], 1), hits[k], totals[k]) for k in hits}


print('quick_pointing_game_eval, quick_recall1_eval defined.')


In [ ]:
# ── 9. Training loop ─────────────────────────────────────────────────────────

def set_seed(seed):
    """Seed Python / NumPy / torch (CPU + CUDA) so runs are reproducible up to
    cuDNN non-determinism. Pair with a torch.Generator passed to DataLoader to
    fix the shuffle order — that's the dominant variance source in our LoRA
    finetune (5pp PG spread observed across v2, v4 prev, v4 this).
    """
    import random as _random
    _random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def train_one_epoch(model, processor, train_ds, lambda_region,
                    optimizer, scheduler, device, cfg,
                    run_tag='', step_offset=0, eval_every=200,
                    dataloader_gen=None):
    model.train()
    loader = DataLoader(
        train_ds,
        batch_size   = cfg['batch_size'],
        shuffle      = True,
        num_workers  = 2,
        collate_fn   = FlickrSigLIPDataset.collate_fn,
        pin_memory   = device == 'cuda',
        generator    = dataloader_gen,
    )

    scaler = torch.amp.GradScaler('cuda', enabled=(device == 'cuda'))

    total_loss = total_global = total_region = 0.0
    n_steps = 0

    pbar = tqdm(loader, desc=f'train {run_tag}')
    for pils, captions, orig_sizes, phrase_boxes_batch in pbar:

        # ── L_global: image ↔ caption ────────────────────────────────────────
        img_inputs = processor(images=pils, return_tensors='pt',
                               padding=True).to(device)
        txt_inputs = processor(text=captions, return_tensors='pt',
                               padding=True, truncation=True).to(device)

        with torch.amp.autocast('cuda', enabled=(device == 'cuda')):
            img_feats = get_image_feats(model, img_inputs['pixel_values'])
            txt_feats = get_text_feats(model, txt_inputs['input_ids'])
            l_global  = siglip_global_loss(
                img_feats, txt_feats,
                model.logit_scale, model.logit_bias
            )

        # ── L_region: phrase ↔ bbox patches ──────────────────────────────────
        l_region = torch.tensor(0.0, device=device)

        if lambda_region > 0:
            triples = []
            for img_idx, pb_list in enumerate(phrase_boxes_batch):
                for phrase, bbox in pb_list:
                    triples.append((img_idx, phrase, bbox))

            if len(triples) >= 2:
                random.shuffle(triples)
                triples = triples[:cfg['region_batch']]

                reg_img_idxs = [t[0] for t in triples]
                reg_phrases  = [t[1] for t in triples]
                reg_bboxes   = [t[2] for t in triples]
                reg_orig_sz  = [orig_sizes[i] for i in reg_img_idxs]
                reg_pils     = [pils[i] for i in reg_img_idxs]
                reg_pix      = processor(images=reg_pils, return_tensors='pt',
                                         padding=True).pixel_values.to(device)

                with torch.amp.autocast('cuda', enabled=(device == 'cuda')):
                    patch_feats_n    = get_patch_feats(model, reg_pix)
                    phrase_tok_feats = get_phrase_token_feats(
                        model, processor, reg_phrases, device
                    )
                    bbox_masks = torch.stack([
                        bbox_patch_mask(bbox, sz, cfg['n_side'], cfg['eval_size'])
                        for bbox, sz in zip(reg_bboxes, reg_orig_sz)
                    ]).to(device)
                    l_region = filip_region_loss(
                        patch_feats_n, phrase_tok_feats, bbox_masks,
                        model.logit_scale, model.logit_bias
                    )

        loss = l_global + lambda_region * l_region

        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        total_loss   += loss.item()
        total_global += l_global.item()
        total_region += l_region.item() if isinstance(l_region, torch.Tensor) else 0.0
        n_steps      += 1
        global_step   = step_offset + n_steps

        wandb.log({
            f'{run_tag}/loss':         loss.item(),
            f'{run_tag}/loss_global':  l_global.item(),
            f'{run_tag}/loss_region':  l_region.item() if isinstance(l_region, torch.Tensor) else 0.0,
            f'{run_tag}/lr':           scheduler.get_last_lr()[0],
            f'{run_tag}/logit_scale':  model.logit_scale.item(),
            f'{run_tag}/logit_bias':   model.logit_bias.item(),
        }, step=global_step)

        pbar.set_postfix({
            'loss':   f'{total_loss/n_steps:.3f}',
            'global': f'{total_global/n_steps:.3f}',
            'region': f'{total_region/n_steps:.3f}',
        })

        # ── Periodic Pointing Game eval (primary metric) ──────────────────────
        if eval_every > 0 and n_steps % eval_every == 0:
            pg_acc, pg_correct, pg_total = quick_pointing_game_eval(
                model, processor,
                n_images = cfg['eval_images'],
                seed     = 42,
                device   = device,
            )
            wandb.log({f'{run_tag}/pointing_game': pg_acc}, step=global_step)
            pbar.write(f'  step {global_step:4d}  Pointing Game = {pg_acc:.2f}%  '
                       f'({pg_correct}/{pg_total})  [B0=14.74%]')
            model.train()   # quick_pointing_game_eval calls model.eval(); restore

    stats = {
        'loss':   total_loss   / n_steps,
        'global': total_global / n_steps,
        'region': total_region / n_steps,
    }
    return stats, n_steps


def make_optimizer_scheduler(model, n_steps, cfg, restarts=1):
    """AdamW + LambdaLR. With restarts=1 (default), a single warmup→cosine→0
    schedule covers all n_steps (used by v2, v3). With restarts>1, the schedule
    is divided into `restarts` equal cycles, each its own warmup→cosine→0 —
    every epoch gets a fresh peak LR and a tight late-cycle settle window.
    This is the v3 → v4 fix (Exp-05): v3 stretched the cosine over 2 epochs
    and the LR never decayed enough to let the model settle into v2's basin.
    """
    optimizer = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=cfg['lr'], weight_decay=cfg['weight_decay']
    )
    steps_per_cycle = max(1, n_steps // max(1, restarts))
    warmup          = cfg['warmup_steps']
    def lr_lambda(step):
        cycle_step = step % steps_per_cycle
        if cycle_step < warmup:
            return cycle_step / max(1, warmup)
        progress = (cycle_step - warmup) / max(1, steps_per_cycle - warmup)
        return max(0.0, 0.5 * (1 + np.cos(np.pi * progress)))
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    return optimizer, scheduler


print('Training loop defined.')


In [ ]:
# ── 10. Train M_human ────────────────────────────────────────────────────────
# Mode toggles. Same cell runs v5 variance study or v6+ leader-seed sweeps —
# edit the three constants below.
#
#   v5 (variance, Exp-06):   SEEDS=[0,1,2], LAMBDA_OVERRIDE=None, TAG='v5'
#   v6 (λ sweep,  Exp-07):   SEEDS=[2],     LAMBDA_OVERRIDE=1.0,  TAG='v6'
#
# v5 found PG = 18.68 ± 2.57% across 3 seeds at λ=0.5. Leader seed (best of 3)
# is seed=2 at 20.90%. We iterate on seed=2 alone for speed (~40 min/run vs
# ~120 min/full-variance) and re-verify any winning recipe across all 3
# seeds at the end.

SEEDS           = [2]       # leader-seed mode for v6 iteration
LAMBDA_OVERRIDE = 1.0       # v6: push region-loss weight to sharpen PG argmax
EXPERIMENT_TAG  = 'v6'      # used in W&B run name + checkpoint path
CFG['epochs']   = 1         # match v2; single cosine, restarts=1.

if LAMBDA_OVERRIDE is not None:
    CFG['lambda_region'] = LAMBDA_OVERRIDE
print(f'Running {EXPERIMENT_TAG} | seeds={SEEDS} | λ_region={CFG["lambda_region"]}')

# Free any leftover model from a previous run before the loop.
import gc
for _var in ['model_mh', 'opt_mh', 'sch_mh']:
    if _var in globals():
        del globals()[_var]
gc.collect()
torch.cuda.empty_cache()
print(f'GPU free at start: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB')

v5_results = []

for seed in SEEDS:
    print(f'\n{"="*72}\n  M_human v5 — seed={seed}\n{"="*72}')
    set_seed(seed)

    # Fresh W&B run per seed.
    if wandb.run is not None:
        wandb.finish()
    mh_run = wandb.init(
        project = 'region-grounded',
        name    = f'm_human_{EXPERIMENT_TAG}_seed{seed}_lam{CFG["lambda_region"]}',
        config  = {**CFG, 'variant': f'm_human_{EXPERIMENT_TAG}', 'seed': seed,
                   'lr_schedule': 'cosine_single'},
        tags    = ['siglip', 'flickr30k', 'lora', 'm_human', EXPERIMENT_TAG,
                   f'seed{seed}', f'lam{CFG["lambda_region"]}'],
        reinit  = True,
    )
    print(f'  W&B: {mh_run.url}')

    # Fresh model + LoRA per seed.
    if 'model_mh' in globals():
        del model_mh
        gc.collect()
        torch.cuda.empty_cache()
    base_model_mh = SiglipModel.from_pretrained(CFG['model_id'],
                                                 token=os.environ['HF_TOKEN'])
    model_mh = get_peft_model(base_model_mh, lora_cfg).to(DEVICE)
    for n, p in model_mh.named_parameters():
        if 'logit_scale' in n or 'logit_bias' in n:
            p.requires_grad_(True)
    model_mh.enable_input_require_grads()
    model_mh.gradient_checkpointing_enable()

    steps_per_epoch = len(train_ds) // CFG['batch_size']
    n_steps_mh     = steps_per_epoch * CFG['epochs']
    opt_mh, sch_mh = make_optimizer_scheduler(
        model_mh, n_steps_mh, CFG, restarts=1,
    )

    # MaskCLIP bypass — train and eval read the same patch representation.
    orig_attn_fwd_mh = install_maskclip_bypass(model_mh)

    # Step-0 sanity (deterministic — just B0 reload with bypass).
    pg0, pg0_c, pg0_t = quick_pointing_game_eval(
        model_mh, processor, n_images=CFG['eval_images'], seed=42, device=DEVICE,
    )
    print(f'  Seed {seed} step 0 PG: {pg0:.2f}%  ({pg0_c}/{pg0_t})')
    wandb.log({'mhuman/pointing_game': pg0}, step=0)

    # Deterministic DataLoader shuffle: torch.Generator seeded explicitly.
    dl_gen = torch.Generator()
    dl_gen.manual_seed(seed)

    stats_ep, ep_steps = train_one_epoch(
        model_mh, processor, train_ds,
        lambda_region = CFG['lambda_region'],
        optimizer     = opt_mh,
        scheduler     = sch_mh,
        device        = DEVICE,
        cfg           = CFG,
        run_tag       = 'mhuman',
        step_offset   = 0,
        eval_every    = 200,
        dataloader_gen= dl_gen,
    )

    # Final PG + R@1 on the same val protocol as B0.
    pg_acc, pg_c, pg_t = quick_pointing_game_eval(
        model_mh, processor, n_images=CFG['eval_images'], seed=42, device=DEVICE,
    )
    r1 = quick_recall1_eval(
        model_mh, processor, n_images=CFG['eval_images'], seed=42, device=DEVICE,
    )

    model_mh.save_pretrained(
        f'{CFG["ckpt_dir"]}/mhuman_{EXPERIMENT_TAG}_seed{seed}_lam{CFG["lambda_region"]}'
    )
    restore_attn(model_mh, orig_attn_fwd_mh)

    v5_results.append({
        'seed':            seed,
        'pg_acc':          pg_acc,
        'recall1_top10':   r1['top10'][0],
        'recall1_top25':   r1['top25'][0],
        'recall1_halfmax': r1['halfmax'][0],
        'loss':            stats_ep['loss'],
        'global':          stats_ep['global'],
        'region':          stats_ep['region'],
    })

    wandb.log({
        'eval/mhuman_pointing_game'    : pg_acc,
        'eval/mhuman_recall1_top10'    : r1['top10'][0],
        'eval/mhuman_recall1_top25'    : r1['top25'][0],
        'eval/mhuman_recall1_halfmax'  : r1['halfmax'][0],
        'mhuman/final_loss'            : stats_ep['loss'],
        'mhuman/final_loss_global'     : stats_ep['global'],
        'mhuman/final_loss_region'     : stats_ep['region'],
    }, step=ep_steps)
    wandb.summary.update({
        'seed'                : seed,
        'pointing_game'       : pg_acc,
        'recall1_top10'       : r1['top10'][0],
        'recall1_top25'       : r1['top25'][0],
        'recall1_halfmax'     : r1['halfmax'][0],
    })
    wandb.finish()

    print(f'  Seed {seed} final PG: {pg_acc:.2f}%  ({pg_c}/{pg_t})')

# Cross-seed summary (printed here AND aggregated more fully in cell 11).
pg_vals = np.array([r['pg_acc'] for r in v5_results])
print(f'\n{"="*72}\n  {EXPERIMENT_TAG} summary ({len(v5_results)} seed(s), λ={CFG["lambda_region"]})\n{"="*72}')
for r in v5_results:
    print(f'  seed {r["seed"]}: PG = {r["pg_acc"]:.2f}%')
if len(pg_vals) >= 2:
    print(f'  mean ± std : {pg_vals.mean():.2f} ± {pg_vals.std(ddof=1):.2f}%')
    print(f'  range      : [{pg_vals.min():.2f}, {pg_vals.max():.2f}]')
else:
    print(f'  single seed (no std). vs v5 seed{v5_results[0]["seed"]} baseline 20.90%: '
          f'Δ = {v5_results[0]["pg_acc"] - 20.90:+.2f}pp')


In [ ]:
# ── 11. Results — per-seed table + reference deltas ───────────────────────────
# Reads v5_results (per-seed dicts) populated by cell 10. Works for any
# EXPERIMENT_TAG ('v5' variance study with n=3, 'v6' leader-seed with n=1,
# or future sweeps). Single-seed runs are compared against v5's leader
# (seed=2 @ λ=0.5, PG=20.90%); multi-seed runs print mean ± std and compare
# against v5's full distribution.
B0 = dict(
    pointing_game   = 14.74,
    recall1_top10   =  6.83,
    recall1_top25   =  7.91,
    recall1_halfmax =  7.58,
    voc_miou        =  1.43,
)
MH_V5 = dict(   # locked v5 baseline (Exp-06, λ=0.5, n=3 seeds)
    pg_mean         = 18.68,
    pg_std          =  2.57,
    pg_seed2_leader = 20.90,
    recall1_top10   =  5.76,
    recall1_top25   =  7.12,
    recall1_halfmax =  7.30,
)

pg_vals  = np.array([r['pg_acc']          for r in v5_results])
r10_vals = np.array([r['recall1_top10']   for r in v5_results])
r25_vals = np.array([r['recall1_top25']   for r in v5_results])
rhm_vals = np.array([r['recall1_halfmax'] for r in v5_results])

print('=' * 96)
print(f'  M_human {EXPERIMENT_TAG}  —  {len(v5_results)} seed(s), 1 epoch, λ={CFG["lambda_region"]}')
print('=' * 96)
print(f'  {"seed":>5} | {"PG":>8} | {"R@1 top-10":>12} | {"R@1 top-25":>12} | {"R@1 halfmax":>13} | {"loss":>7}')
print('-' * 96)
for r in v5_results:
    print(f'  {r["seed"]:>5} | {r["pg_acc"]:7.2f}% | {r["recall1_top10"]:11.2f}% | '
          f'{r["recall1_top25"]:11.2f}% | {r["recall1_halfmax"]:12.2f}% | {r["loss"]:7.3f}')
print('-' * 96)

pg_mean = float(pg_vals.mean())
if len(pg_vals) >= 2:
    pg_std = float(pg_vals.std(ddof=1))
    def _agg(vals):
        return f'{vals.mean():.2f} ± {vals.std(ddof=1):.2f}'
    print(f'  mean± | {_agg(pg_vals):>7}% | {_agg(r10_vals):>11}% | '
          f'{_agg(r25_vals):>11}% | {_agg(rhm_vals):>12}% |')
    print(f'  range | [{pg_vals.min():.2f}, {pg_vals.max():.2f}]  '
          f'(spread = {pg_vals.max() - pg_vals.min():.2f}pp)')
else:
    pg_std = float('nan')
    print(f'  (single seed — no std)')
print('=' * 96)

print(f'\nReference comparisons:')
print(f'  B0 (frozen)            : {B0["pointing_game"]:.2f}%')
print(f'  v5 mean ± std (λ=0.5)  : {MH_V5["pg_mean"]:.2f} ± {MH_V5["pg_std"]:.2f}%')
print(f'  v5 seed=2 leader (λ=0.5): {MH_V5["pg_seed2_leader"]:.2f}%')
print(f'  this run PG            : {pg_mean:.2f}%' +
      (f' ± {pg_std:.2f}' if len(pg_vals) >= 2 else ''))
print(f'  Δ vs B0                : {pg_mean - B0["pointing_game"]:+.2f}pp')
print(f'  Δ vs v5 mean           : {pg_mean - MH_V5["pg_mean"]:+.2f}pp')
print(f'  Δ vs v5 seed=2 leader  : {pg_mean - MH_V5["pg_seed2_leader"]:+.2f}pp')
# Significance call: leader-seed run is "real win" only if it beats v5 mean by > σ_v5.
if len(pg_vals) == 1:
    real_win = pg_mean > MH_V5['pg_mean'] + MH_V5['pg_std']
    print(f'\n>> Single-seed gain > σ_v5 (noise floor)? '
          f'{"YES — re-verify on all 3 seeds" if real_win else "NO (within noise)"}')

# Aggregate W&B run capturing this experiment's headline numbers.
if wandb.run is not None:
    wandb.finish()
agg_run = wandb.init(
    project = 'region-grounded',
    name    = f'm_human_{EXPERIMENT_TAG}_aggregate_n{len(v5_results)}_lam{CFG["lambda_region"]}',
    config  = {**CFG, 'variant': f'm_human_{EXPERIMENT_TAG}_aggregate', 'seeds': SEEDS},
    tags    = ['siglip', 'flickr30k', 'lora', 'm_human', EXPERIMENT_TAG, 'aggregate'],
    reinit  = True,
)
wandb.summary.update({
    'experiment_tag'    : EXPERIMENT_TAG,
    'lambda_region'     : CFG['lambda_region'],
    'seeds_n'           : len(v5_results),
    'pg_mean'           : pg_mean,
    'pg_std'            : pg_std,
    'pg_min'            : float(pg_vals.min()),
    'pg_max'            : float(pg_vals.max()),
    'b0_pointing_game'  : B0['pointing_game'],
    'v5_pg_mean'        : MH_V5['pg_mean'],
    'v5_pg_seed2_leader': MH_V5['pg_seed2_leader'],
    'delta_vs_b0'       : pg_mean - B0['pointing_game'],
    'delta_vs_v5_mean'  : pg_mean - MH_V5['pg_mean'],
    'delta_vs_v5_leader': pg_mean - MH_V5['pg_seed2_leader'],
})
wandb.finish()
print('Aggregate W&B run finished.')
